# PartnerCopilot: verificacao JSON congelada
Novo notebook, kernel novo. GPU + Internet + Secret HF_TOKEN.
Somente dez perguntas NOVAS, tres negativas. Nao executa o prompt nas 30 antigas.
Sem retry ou ajuste ao ver as respostas. O corpus ja e conhecido; nao e holdout
de documentos nem gold humano independente. Nenhuma inferencia real foi feita
na preparacao local. O codigo precisa estar publicado no GitHub.

In [ ]:
import os, sys, subprocess, json
from pathlib import Path
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
import torch
assert torch.cuda.is_available(), 'Ative GPU.'
assert torch.cuda.device_count() == 1, 'Reinicie o kernel para aplicar CUDA_VISIBLE_DEVICES.'
print(torch.cuda.get_device_name(0))
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
assert os.environ['HF_TOKEN'], 'Habilite HF_TOKEN para este notebook.'
os.environ['EMBEDDING_DEVICE'] = 'cpu'

## Instalar codigo e snapshot
Checkout separado; nao altera notebooks v1/v2/D. Nao atualize automaticamente ao
retomar: commit e parametros fazem parte da identidade da rodada.

In [ ]:
repo = Path('/kaggle/working/partner-copilot-json')
if not repo.exists():
    subprocess.run(['git', 'clone', 'https://github.com/JoaoNordemann/partner-copilot.git', str(repo)], check=True)
os.chdir(repo)
assert Path('scripts/evaluate_structured_generation.py').is_file(), 'Publique o codigo JSON no GitHub primeiro.'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=True)
print('Commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())

## Validar antes da inferencia
Esperado: dez perguntas, tres negativas, corpus 100, prompt e parser JSON, retry false.

In [ ]:
output = Path('/kaggle/working/PartnerCopilot/json-verification-v1/generation-json-v1.jsonl')
command = [sys.executable, 'scripts/evaluate_structured_generation.py', '--output', str(output)]
subprocess.run(command + ['--check-only'], check=True)

## Executar uma unica rodada
No maximo uma chamada por pergunta com contexto. Contexto vazio abstem sem LLM.
Nao modifique prompt, parser, modelo ou parametros apos ver as saidas.

In [ ]:
subprocess.run(command + ['--resume'], check=True)

## Medir e ler as dez respostas
Schema, abstencao, IDs no contexto e sustentacao factual sao medidas diferentes.
O ultimo campo fica pendente para leitura: o parser nao conhece a verdade factual.
O template de revisao existente nao e sobrescrito ao recalcular o resumo.

In [ ]:
subprocess.run(command + ['--summarize-only'], check=True)
print(output.with_suffix('.summary.json').read_text(encoding='utf-8'))
print(output.with_suffix('.review.md').read_text(encoding='utf-8'))

## Preservar antes de encerrar
Baixe o ZIP ou salve os outputs do Kaggle. Mantenha esta rodada separada de v1/v2/D.

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from IPython.display import FileLink, display
zip_path = Path('/kaggle/working/partner-copilot-json-verification-v1.zip')
with ZipFile(zip_path, 'w', ZIP_DEFLATED) as archive:
    for path in output.parent.iterdir():
        if path.is_file() and path.suffix != '.lock':
            archive.write(path, arcname=path.name)
display(FileLink(str(zip_path)))